In [38]:
from fem2d import SimpleFrame
from fem2d.results import Results
import numpy as np
frame = SimpleFrame()

frame.add_node(1, 0, 0)
frame.add_node(2, 0, 6)
frame.add_node(3, 0, 12)
frame.add_node(4, 9, 6)
frame.add_node(5, 9, 0)

E = 30e6
I = 4.8e-4
A = 75e-3

frame.add_frame(1, 1, 2, E, A, I)
frame.add_frame(2, 2, 3, E, A, I)
frame.add_frame(3, 3, 4, E, A, I)
frame.add_frame(4, 2, 4, E, A, I)
frame.add_frame(5, 4, 5, E, A, I)

frame.add_support(1, [1, 1, 1])
frame.add_support(5, [1, 1, 1])

frame.add_node_load(2, [80, 0, 0])  # point load
frame.add_node_load(3, [40, 0, 0])
frame.add_distributed_load(3, wy=12)  # udl

frame.solve();


In [39]:
results = Results(frame)
disp = results.node_displacements()
reactions = results.reactions()
el_forces = results.element_forces()

print("Node Displacements:\n", disp)
print("Reactions:\n", reactions)
print("Element Forces:\n", el_forces)

# print(disp["theta"][2])  # result = 0.01789119564872733
# print(el_forces["m_i"][2])

Node Displacements:
    node        ux        uy     theta
0     1  0.000000  0.000000  0.000000
1     2  0.185424  0.000419 -0.017620
2     3  0.186623  0.000714  0.017891
3     4  0.185521 -0.000131 -0.026028
4     5  0.000000  0.000000  0.000000
Reactions:
    node          Fx          Fy           M
0     1 -106.051223 -157.027105  360.441368
1     2    0.000000    0.000000    0.000000
2     3    0.000000    0.000000    0.000000
3     4    0.000000    0.000000    0.000000
4     5  -85.948777   49.027105  320.314685
Element Forces:
    element        fx_i        fy_i         m_i        fx_j        fy_j  \
0        1 -157.027105  106.051223  360.441368  157.027105 -106.051223   
1        2 -110.599106    1.610942  -80.393742  110.599106   -1.610942   
2        3   57.290973  -59.829678  -90.059395 -129.290973  -48.170322   
3        4  -24.440281  -46.427999 -195.472227   24.440281   46.427999   
4        5   49.027105   85.948777  195.377977  -49.027105  -85.948777   

          m_j

In [40]:
k5 = frame.structure.elements[3].local_stiffness()
T = frame.structure.elements[3].transformation_matrix()
T

array([[ 0.83205029, -0.5547002 ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.5547002 ,  0.83205029,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.83205029, -0.5547002 ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.5547002 ,  0.83205029,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ]])

In [41]:
k5

array([[ 2.08012574e+05,  0.00000000e+00,  0.00000000e+00,
        -2.08012574e+05,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  1.36541587e+02,  7.38461538e+02,
         0.00000000e+00, -1.36541587e+02,  7.38461538e+02],
       [ 0.00000000e+00,  7.38461538e+02,  5.32512188e+03,
         0.00000000e+00, -7.38461538e+02,  2.66256094e+03],
       [-2.08012574e+05,  0.00000000e+00,  0.00000000e+00,
         2.08012574e+05,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00, -1.36541587e+02, -7.38461538e+02,
         0.00000000e+00,  1.36541587e+02, -7.38461538e+02],
       [ 0.00000000e+00,  7.38461538e+02,  2.66256094e+03,
         0.00000000e+00, -7.38461538e+02,  5.32512188e+03]])

[6, 7, 8]

In [50]:
ui = frame.structure.disp[frame.structure.elements[3].node_i.dofs]
uj = frame.structure.disp[frame.structure.elements[3].node_j.dofs]
u = np.concatenate([ui, uj])
u

array([ 1.86623368e-01,  7.13669896e-04,  1.78911956e-02,  1.85521414e-01,
       -1.30738947e-04, -2.60284808e-02])

In [32]:
u = frame.structure.disp[6:12]
u

array([ 1.86623368e-01,  7.13669896e-04,  1.78911956e-02,  1.85521414e-01,
       -1.30738947e-04, -2.60284808e-02])

In [34]:
u5 = np.matmul(T, u)
u5

array([ 0.15488416,  0.10411383,  0.0178912 ,  0.15443567,  0.10279998,
       -0.02602848])

In [35]:
np.matmul(k5, u5)

array([ 93.29097262,  -5.82967772,  26.94060469, -93.29097262,
         5.82967772, -89.99821052])

In [51]:
frame.structure.elements[3].eq_load

array([  36.,   54.,  117.,   36.,   54., -117.])

In [52]:
np.matmul(k5, u5) - frame.structure.elements[3].eq_load

array([  57.29097262,  -59.82967772,  -90.05939531, -129.29097262,
        -48.17032228,   27.00178948])